# 05 · Training: Loss Function, Baseline, dan Model Utama
**Bab III Subbab 3.7 dan Tabel 3.5.** Semua model pada notebook ini dilatih pada skala utama (`cfg.main_scale`) dengan target T1.

| Kode | Konfigurasi |
|---|---|
| B0 | urutan berdasarkan panjang isi ulasan |
| B1 | TF-IDF + fitur, ranker linear *pairwise* |
| B2 | TF-IDF (SVD 768) + fitur, MLP *pairwise* (opsional) |
| B3 | BLaIR + fitur, ranker linear *pairwise* (opsional) |
| B4 | BLaIR + fitur, MLP *pointwise* MSE |
| B5 | RoBERTa-base beku + fitur, MLP *pairwise* |
| U | BLaIR + fitur, MLP RankNet (model utama) |

Data uji **tidak** dievaluasi di sini. Skor uji disimpan, tetapi metriknya baru dihitung di notebook 07.

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
import torch, matplotlib.pyplot as plt
from hr import experiment as X, features, embed, plots
from hr.config import EXPERIMENTS
from hr.pairs import PairSampler
from hr.train import evaluate_scores
b = X.load_bundle(cfg)
MS, TG = cfg.main_scale, cfg.main_target
todo = cfg.experiments_to_run()
tr, va, te = b.rows("train", MS), b.rows("val"), b.rows("test")
print("latih", len(tr), "| validasi", len(va), "| uji", len(te), "| eksperimen:", todo)

## 1. Loss function
**RankNet (persamaan 2.12–2.14, 3.6).** Untuk pasangan ulasan dari produk yang sama dengan $y_i > y_j$,
$C_{ij} = \ln(1 + e^{-(s_i - s_j)})$. Loss kecil jika model memberi skor $s_i$ jauh di atas $s_j$, dan membesar jika urutannya terbalik.
**Pointwise MSE (B4)** menaksir $y_i$ langsung tanpa melihat ulasan lain.

In [ ]:
d = np.linspace(-5, 5, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(d, np.log1p(np.exp(-d))); ax[0].set_xlabel("$s_i - s_j$"); ax[0].set_ylabel("loss RankNet")
ax[0].set_title("RankNet: menghukum urutan terbalik")
ax[1].plot(d, d ** 2); ax[1].set_xlabel("$s_i - y_i$"); ax[1].set_ylabel("loss MSE"); ax[1].set_title("Pointwise MSE")
fig.tight_layout(); plots.save(fig, cfg.path("figures", "loss_function.png"))

## 2. Pasangan pelatihan (persamaan 3.5)

In [ ]:
pair_tab = []
for M in sorted({20, 50, cfg.max_pairs_per_product, 200}):
    s = PairSampler(tr["parent_asin"].to_numpy(), tr["y_t1"].to_numpy(), M, cfg.pair_delta, seed=cfg.seed)
    pair_tab.append({"M": M, **s.stats(tr["parent_asin"].to_numpy())})
pair_tab = pd.DataFrame(pair_tab); pair_tab.to_csv(cfg.path("tables", "pasangan.csv"), index=False); pair_tab

## 3. B0 · panjang ulasan

In [ ]:
def val_metrics(scores, rows=va, target=TG):
    es = b.eval_set(rows, np.zeros((len(rows), 1)), target)
    return evaluate_scores(es, scores, cfg.map_relevant_grade)[0]
m_b0 = val_metrics(features.score_b0(va))
X.save_scores(cfg, X.run_id("B0", MS, TG, 0), {"code": "B0", "scale": MS, "target": TG, "seed": 0,
              "best_val_ndcg@10": m_b0["ndcg@10"]}, va, features.score_b0(va), te, features.score_b0(te))
m_b0

## 4. B1 · TF-IDF + fitur, ranker linear
Regresi logistik pada selisih vektor pasangan adalah RankNet dengan pemberi skor linear. Parameter C ditala pada data validasi.

In [ ]:
if "B1" in todo:
    vec = embed.fit_tfidf(embed.review_text(tr), max_features=200_000 if not SMOKE else 2000)
    Xs = {n: features.tfidf_plus_struct(vec, embed.review_text(d), d) for n, d in (("tr", tr), ("va", va), ("te", te))}
    tune = []
    for C in cfg.linear_c_grid:
        clf = features.train_b1(Xs["tr"], tr["y_t1"].to_numpy(), tr["parent_asin"].to_numpy(), C,
                                cfg.max_pairs_per_product, cfg.pair_delta, cfg.seed)
        tune.append({"C": C, **val_metrics(features.score_linear(clf, Xs["va"]))})
    tune = pd.DataFrame(tune); display(tune)
    C_best = float(tune.loc[tune["ndcg@10"].idxmax(), "C"])
    clf = features.train_b1(Xs["tr"], tr["y_t1"].to_numpy(), tr["parent_asin"].to_numpy(), C_best,
                            cfg.max_pairs_per_product, cfg.pair_delta, cfg.seed)
    X.save_scores(cfg, X.run_id("B1", MS, TG, cfg.seed), {"code": "B1", "scale": MS, "target": TG, "seed": cfg.seed,
                  "C": C_best, "best_val_ndcg@10": float(tune["ndcg@10"].max())},
                  va, features.score_linear(clf, Xs["va"]), te, features.score_linear(clf, Xs["te"]))

## 5. Penalaan hiperparameter model utama U
Grid kecil *learning rate* × *dropout* pada data validasi dengan satu seed. Hiperparameter terbaik dipakai untuk semua model MLP agar perbandingannya adil.

In [ ]:
tune_rows = []
for lr in cfg.lr_grid:
    for do in cfg.dropout_grid:
        r = X.run_torch(b, "U", scale=MS, target=TG, seed=cfg.seed, lr=lr, dropout=do,
                        extra=f"tune_lr{lr}_do{do}", verbose=False)
        tune_rows.append({"lr": lr, "dropout": do, "val_ndcg@10": r["best_val_ndcg@10"], "epochs": r["epochs_run"]})
tune_tab = pd.DataFrame(tune_rows).sort_values("val_ndcg@10", ascending=False)
tune_tab.to_csv(cfg.path("tables", "penalaan_U.csv"), index=False)
best_hp = {"lr": float(tune_tab.iloc[0]["lr"]), "dropout": float(tune_tab.iloc[0]["dropout"])}
cfg.path("best_hparams.json").write_text(json.dumps(best_hp))
display(tune_tab); best_hp

## 6. Training akhir dengan beberapa seed

In [ ]:
if "B2" in todo:   # SVD 768 dari TF-IDF, disimpan sejajar dengan berkas embedding
    if not cfg.path("emb_tfidf_svd.npy").exists():
        svd = embed.fit_svd(Xs["tr"][:, :-5], n_components=768 if not SMOKE else 16, seed=cfg.seed)
        allrows = b.df.set_index("rid").loc[b.pos.index].reset_index()
        Z = svd.transform(vec.transform(embed.review_text(allrows))).astype(np.float32)
        Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9
        np.save(cfg.path("emb_tfidf_svd.npy"), Z.astype(np.float16))
        cfg.path("emb_tfidf_svd.json").write_text(json.dumps({"done": len(Z), "rows": len(Z), "model": None,
            "fingerprint": embed.rid_fingerprint(b.pos.index.to_numpy()), "pooling": "tfidf+svd+l2"}))
results = []
for code in ["U", "B4", "B5", "B3", "B2"]:
    if code not in todo:
        continue
    for seed in cfg.seeds:
        print(f"--- {code} seed {seed}")
        results.append(X.run_torch(b, code, scale=MS, target=TG, seed=seed, **best_hp))

## 7. Ringkasan validasi

In [ ]:
val_tab = X.validation_table(cfg, MS, TG)
val_tab.to_csv(cfg.path("tables", "validasi_baseline.csv")); val_tab

In [ ]:
h = pd.read_csv(cfg.out_dir / "runs" / X.run_id("U", MS, TG, cfg.seeds[0]) / "history.csv")
axes = plots.loss_curves(h, "Model utama U")
plots.save(axes[0].figure, cfg.path("figures", "loss_U.png")); h

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()